# 🏥 KG-CMI: Knowledge Graph Enhanced Cross-Mamba Interaction for Med-VQA
### 🎓 Minor Project Training Notebook (Google Colab / Cloud GPU)

## Step 1: Hardware Check & Repository Cloning

In [ ]:
!nvidia-smi
import os, shutil
if os.path.exists('/content/Med-VQA'):
    shutil.rmtree('/content/Med-VQA')
!git clone https://github.com/Anjali-2807/Med-VQA.git || git clone https://github.com/BioMedIA-repo/KG-CMI.git Med-VQA
%cd Med-VQA
print('✅ Repository cloned!')

## Step 2: Install All Required Dependencies (exact pinned versions)

In [ ]:
# Pin ALL packages to exact versions from requirements.txt for full compatibility
!pip install -q timm==0.4.12
!pip install -q pytorch-lightning==1.9.5
!pip install -q transformers==4.35.2
!pip install -q einops sacred pandas ftfy torchmetrics==0.7.2 scikit-learn
!pip install -q torch-geometric pyarrow
print('✅ All dependencies installed!')

## Step 2.5: Apply Compatibility Patches

In [ ]:
import re

# Patch: Fix tokenizer_class in bert_model.py (removed in transformers 4.12+)
bert_path = '/content/Med-VQA/m3ae/modules/language_encoders/bert_model.py'
with open(bert_path, 'r') as f:
    content = f.read()
content = re.sub(r'\s*tokenizer_class=_TOKENIZER_FOR_DOC,', '', content)
with open(bert_path, 'w') as f:
    f.write(content)
print('✅ bert_model.py patched!')

## Step 3: Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoints will be saved to: {CHECKPOINT_DIR}')

## Step 4: Verify Environment & Inject Upgraded R-GCN Module

In [ ]:
import sys
sys.path.insert(0, '/content/Med-VQA')

import torch
import pytorch_lightning as pl
import timm
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA available: {torch.cuda.is_available()}')
print(f'✅ PyTorch Lightning: {pl.__version__}')
print(f'✅ timm: {timm.__version__}')

import torch.nn as nn

class RelationalGraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features, num_relations=4, num_bases=4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_relations = num_relations
        self.num_bases = num_bases
        self.weight_self = nn.Parameter(torch.Tensor(in_features, out_features))
        self.bases = nn.Parameter(torch.Tensor(num_bases, in_features, out_features))
        self.rel_coeffs = nn.Parameter(torch.Tensor(num_relations, num_bases))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_self)
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.rel_coeffs)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type):
        N_v = x.size(0)
        out = torch.matmul(x, self.weight_self)
        W_r = torch.einsum('rb, bio -> rio', self.rel_coeffs, self.bases)
        src, dst = edge_index[0], edge_index[1]
        for r in range(self.num_relations):
            rel_mask = (edge_type == r)
            if not rel_mask.any():
                continue
            r_src, r_dst = src[rel_mask], dst[rel_mask]
            msg = torch.matmul(x[r_src], W_r[r])
            deg = torch.zeros(N_v, device=x.device).scatter_add_(0, r_dst, torch.ones_like(r_dst, dtype=torch.float)).clamp(min=1.0)
            norm = 1.0 / deg[r_dst].unsqueeze(-1)
            out.scatter_add_(0, r_dst.unsqueeze(-1).expand(-1, self.out_features), msg * norm)
        return out + self.bias

print('✅ Upgraded R-GCN module ready!')

## Step 5: Start Model Training

In [ ]:
!python main.py \
    --config ./lavis/projects/blip2/train/vqa_rad.yaml \
    --output_dir /content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints